# AIRMAN Training Intelligence Analysis
This notebook loads the synthetic training datasets, validates data quality, computes Skynet operational metrics, analyzes cadet training progress, evaluates TOGA study intelligence, assesses finance risk, and generates explainable risk scores and visualizations.

In [6]:

# Required libraries for analysis
import os
from pathlib import Path
from datetime import timedelta

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Compatibility fix for newer NumPy versions
if not hasattr(np, "matrix"):
    np.matrix = np.asarray

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries loaded successfully")


## Data Loading
Load all required datasets from the `data/` folder and display sample records for validation.

In [8]:

# Load datasets safely

PROJECT_ROOT = Path.cwd()

# If notebook is inside notebooks/ folder, move one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

required_files = [
    'sorties.csv',
    'aircraft.csv',
    'cadets.csv',
    'instructors.csv',
    'toga_study.csv',
    'payments.csv'
]

for file in required_files:
    file_path = DATA_DIR / file
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required file: {file_path}")

sorties = pd.read_csv(
    DATA_DIR / 'sorties.csv',
    parse_dates=['scheduled_start', 'scheduled_end', 'actual_start', 'actual_end']
)

aircraft = pd.read_csv(DATA_DIR / 'aircraft.csv')

cadets = pd.read_csv(
    DATA_DIR / 'cadets.csv',
    parse_dates=['enrollment_date']
)

instructors = pd.read_csv(DATA_DIR / 'instructors.csv')

toga_study = pd.read_csv(
    DATA_DIR / 'toga_study.csv',
    parse_dates=['last_active_date']
)

payments = pd.read_csv(
    DATA_DIR / 'payments.csv',
    parse_dates=['last_payment_date']
)

print("\nDatasets loaded successfully:")
print("Sorties:", sorties.shape)
print("Aircraft:", aircraft.shape)
print("Cadets:", cadets.shape)
print("Instructors:", instructors.shape)
print("TOGA Study:", toga_study.shape)
print("Payments:", payments.shape)


Current Working Directory: C:\Users\Aditri\airman-data-science-assessment

Datasets Loaded Successfully:

Sorties: (200, 13)
Aircraft: (8, 7)
Cadets: (30, 7)
Instructors: (10, 6)
TOGA Study: (90, 7)
Payments: (30, 5)
Sorties 200
Aircraft 8
Cadets 30
Instructors 10
TOGA records 90
Payments 30


,sortie_id,cadet_id,instructor_id,aircraft_id,base_id,scheduled_start,scheduled_end,actual_start,actual_end,status,delay_minutes,cancel_reason,lesson_type
0,S001,C001,I010,A004,B01,2026-05-13,2026-05-13 01:00:00,2026-05-13 00:00:00,2026-05-13 01:00:00,completed,-3,NaN,Navigation
1,S002,C028,I006,A002,B01,2026-03-04,2026-03-04 03:00:00,2026-03-04 00:59:00,2026-03-04 03:59:00,completed,59,NaN,Cross-Country
2,S003,C019,I008,A004,B02,2026-01-30,2026-01-30 01:00:00,2026-01-30 00:45:00,2026-01-30 01:45:00,completed,45,NaN,Instrument Flying
3,S004,C015,I001,A002,B01,2026-01-31,2026-01-31 03:00:00,2026-01-31 00:03:00,2026-01-31 03:03:00,completed,3,NaN,Circuit
4,S005,C018,I008,A003,B02,2026-03-29,2026-03-29 02:00:00,2026-03-29 00:00:00,2026-03-29 02:00:00,completed,-1,NaN,Cross-Country


## Data Cleaning and Validation
Run systematic checks for missing values, duplicate identifiers, invalid statuses, date consistency, payment reconciliation, and progress anomalies.

In [9]:
# Validation helpers
issues = []

# Missing values
for name, df in [('sorties', sorties), ('aircraft', aircraft), ('cadets', cadets), ('instructors', instructors), ('toga_study', toga_study), ('payments', payments)]:
    missing = df.isna().any(axis=1)
    if missing.any():
        issues.append((name, 'Missing values', missing.sum()))

# Duplicate identifiers
for name, column, df in [('sorties', 'sortie_id', sorties), ('aircraft', 'aircraft_id', aircraft), ('cadets', 'cadet_id', cadets), ('instructors', 'instructor_id', instructors)]:
    dup = df[df.duplicated(subset=[column], keep=False)]
    if len(dup) > 0:
        issues.append((name, f'Duplicate {column}', len(dup)))

# Status validation
invalid_status = sorties[~sorties['status'].isin(['completed', 'cancelled', 'delayed'])]
if len(invalid_status) > 0:
    issues.append(('sorties', 'Invalid statuses', len(invalid_status)))

# Date consistency
invalid_schedule = sorties[sorties['scheduled_end'] < sorties['scheduled_start']]
if len(invalid_schedule) > 0:
    issues.append(('sorties', 'Invalid schedule windows', len(invalid_schedule)))
invalid_actual = sorties[sorties['actual_end'] < sorties['actual_start']]
if len(invalid_actual) > 0:
    issues.append(('sorties', 'Invalid actual windows', len(invalid_actual)))

# Negative delays
negative_delay = sorties[sorties['delay_minutes'] < 0]
if len(negative_delay) > 0:
    issues.append(('sorties', 'Negative delay minutes', len(negative_delay)))

# Completed sorties with missing actual times
completed_missing_actual = sorties[(sorties['status'] == 'completed') & sorties[['actual_start', 'actual_end']].isna().any(axis=1)]
if len(completed_missing_actual) > 0:
    issues.append(('sorties', 'Completed sorties missing actual time', len(completed_missing_actual)))

# Cancelled sorties with actual times
cancelled_with_actual = sorties[(sorties['status'] == 'cancelled') & sorties[['actual_start', 'actual_end']].notna().any(axis=1)]
if len(cancelled_with_actual) > 0:
    issues.append(('sorties', 'Cancelled sorties with actual flight times', len(cancelled_with_actual)))

# Payment validation
invalid_payments = payments[payments['invoiced_amount'] != payments['paid_amount'] + payments['outstanding_amount']]
if len(invalid_payments) > 0:
    issues.append(('payments', 'Reconciliation mismatch', len(invalid_payments)))

# Study progress > 100%
over_progress = toga_study[toga_study['chapters_completed'] > toga_study['total_chapters']]
if len(over_progress) > 0:
    issues.append(('toga_study', 'Study progress exceeds 100%', len(over_progress)))

# Downtime validation
downtime_issues = aircraft[aircraft['maintenance_downtime_hours'] > aircraft['total_available_hours']]
if len(downtime_issues) > 0:
    issues.append(('aircraft', 'Downtime exceeds available hours', len(downtime_issues)))

# Flown vs required hours
overflown = cadets[cadets['total_flown_hours'] > cadets['total_required_hours']]
if len(overflown) > 0:
    issues.append(('cadets', 'Flown hours exceed required hours', len(overflown)))

issues

[('sorties', 'Missing values', np.int64(198)),
 ('sorties', 'Negative delay minutes', 18),
 ('sorties', 'Completed sorties missing actual time', 10),
 ('sorties', 'Cancelled sorties with actual flight times', 2),
 ('toga_study', 'Study progress exceeds 100%', 5)]

### Calculated Duration and Delay Consistency
Add a derived actual duration field and verify that delay values match actual start differences.

In [12]:

# Derive duration metrics safely

sorties = sorties.copy()

# Ensure datetime conversion
datetime_cols = [
    'scheduled_start',
    'scheduled_end',
    'actual_start',
    'actual_end'
]

for col in datetime_cols:
    sorties[col] = pd.to_datetime(sorties[col], errors='coerce')

# Actual duration
sorties['actual_duration_hours'] = (
    (sorties['actual_end'] - sorties['actual_start'])
    .dt.total_seconds() / 3600
)

# Scheduled duration
sorties['scheduled_duration_hours'] = (
    (sorties['scheduled_end'] - sorties['scheduled_start'])
    .dt.total_seconds() / 3600
)

# Clean invalid values
sorties['actual_duration_hours'] = (
    sorties['actual_duration_hours']
    .fillna(0)
    .clip(lower=0)
)

sorties['scheduled_duration_hours'] = (
    sorties['scheduled_duration_hours']
    .fillna(0)
    .clip(lower=0)
)

# Delay calculation
if 'delay_minutes' not in sorties.columns:
    sorties['delay_minutes'] = (
        (sorties['actual_start'] - sorties['scheduled_start'])
        .dt.total_seconds() / 60
    )

sorties['delay_minutes'] = sorties['delay_minutes'].fillna(0)

print("Duration and delay metrics created successfully")
print(sorties[['actual_duration_hours', 'scheduled_duration_hours', 'delay_minutes']].head())


Working Directory: C:\Users\Aditri\airman-data-science-assessment


TypeError: isinstance() arg 2 must be a type, a tuple of types, or a union

## Skynet Operations Analytics
Compute aircraft and instructor utilization, workload balance, dispatch reliability, and cancellation trends.

In [13]:
# Aircraft utilization and base-level metrics
actual_flights = sorties[sorties['actual_duration_hours'].notna()].copy()
aircraft_util = actual_flights.groupby('aircraft_id')['actual_duration_hours'].sum().reset_index()
aircraft_util = aircraft_util.merge(aircraft[['aircraft_id', 'registration', 'total_available_hours']], on='aircraft_id')
aircraft_util['utilization_pct'] = (aircraft_util['actual_duration_hours'] / aircraft_util['total_available_hours']) * 100

totals_by_base = actual_flights.groupby('base_id')['actual_duration_hours'].sum().reset_index().rename(columns={'actual_duration_hours': 'base_flight_hours'})

# Instructor utilization and workload
instructor_util = actual_flights.groupby('instructor_id')['actual_duration_hours'].sum().reset_index()
instructor_util = instructor_util.merge(instructors[['instructor_id', 'name', 'total_duty_hours']], on='instructor_id')
instructor_util['utilization_pct'] = (instructor_util['actual_duration_hours'] / instructor_util['total_duty_hours']) * 100
instructor_util['workload_bucket'] = pd.cut(instructor_util['utilization_pct'], bins=[-1, 50, 80, 100, 200], labels=['Underutilized', 'Balanced', 'High', 'Overloaded'])

overloaded = instructor_util[instructor_util['workload_bucket'] == 'Overloaded']
underutilized_aircraft = aircraft_util[aircraft_util['utilization_pct'] < 50]

# Dispatch reliability metrics
completed = sorties[sorties['status'] == 'completed']
delayed = sorties[sorties['status'] == 'delayed']
cancelled = sorties[sorties['status'] == 'cancelled']
total_sorties = len(sorties)

dispatch_reliability = ((len(completed) + len(delayed)) / total_sorties) * 100
completion_rate = (len(completed) / total_sorties) * 100
cancellation_rate = (len(cancelled) / total_sorties) * 100
average_delay = actual_flights['delay_minutes'].replace({np.nan: 0}).mean()

# Robust column naming across all pandas versions
_vc = cancelled['cancel_reason'].value_counts().reset_index()
_vc.columns = ['reason', 'count']
top_cancellation_reasons = _vc

delay_by_lesson = actual_flights.groupby('lesson_type')['delay_minutes'].mean().reset_index().sort_values('delay_minutes', ascending=False)
delay_by_base = actual_flights.groupby('base_id')['delay_minutes'].mean().reset_index().sort_values('delay_minutes', ascending=False)

aircraft_util.head(), instructor_util.head(), dispatch_reliability, completion_rate, cancellation_rate

KeyError: 'actual_duration_hours'

## Training Progress Analytics
Measure cadet progress percentages, remaining flight hours, average acceleration rate, and identify training risk cadets.

In [ ]:
cadets = cadets.copy()
cadets['progress_pct'] = (cadets['total_flown_hours'] / cadets['total_required_hours']) * 100
cadets['remaining_hours'] = (cadets['total_required_hours'] - cadets['total_flown_hours']).clip(lower=0)

cadets['months_enrolled'] = ((pd.Timestamp('2026-05-15') - cadets['enrollment_date']).dt.days / 30).clip(lower=1)
cadets['flying_rate_hpm'] = cadets['total_flown_hours'] / cadets['months_enrolled']

cadets_at_risk = cadets[cadets['progress_pct'] < 60].sort_values('progress_pct')
lesson_disruption = actual_flights.groupby('lesson_type').agg(avg_delay=('delay_minutes', 'mean'), cancellations=('status', lambda x: (x == 'cancelled').sum()), sortie_count=('sortie_id', 'count')).reset_index().sort_values('avg_delay', ascending=False)

instructor_bottlenecks = actual_flights.groupby('instructor_id').agg(total_hours=('actual_duration_hours', 'sum'), average_delay=('delay_minutes', 'mean'), sorties=('sortie_id', 'count')).reset_index().merge(instructors[['instructor_id', 'name']], on='instructor_id').sort_values('total_hours', ascending=False)

cadets_at_risk.head(), lesson_disruption.head(), instructor_bottlenecks.head()

## TOGA Study Intelligence
Compute study readiness, subject weak points, inactivity risk, and recommended actions for cadets.

In [ ]:
toga_study = toga_study.copy()
toga_study['study_progress_pct'] = (toga_study['chapters_completed'] / toga_study['total_chapters']) * 100

subject_summary = toga_study.groupby('subject').agg(avg_progress=('study_progress_pct', 'mean'), avg_quiz=('avg_quiz_score', 'mean')).reset_index().sort_values('avg_progress')

cadet_study = toga_study.groupby('cadet_id').agg(
    subjects=('subject', 'count'),
    avg_progress=('study_progress_pct', 'mean'),
    avg_quiz=('avg_quiz_score', 'mean'),
    last_active=('last_active_date', 'max'),
    practice_tests=('practice_tests_attempted', 'sum')
).reset_index().merge(cadets[['cadet_id', 'name']], on='cadet_id')

cadet_study['inactive_days'] = (pd.Timestamp('2026-05-15') - cadet_study['last_active']).dt.days
cadet_study['inactivity_risk'] = pd.cut(cadet_study['inactive_days'], bins=[-1, 14, 30, 365], labels=['Low', 'Medium', 'High'])

cadet_study['study_readiness'] = (
    cadet_study['avg_quiz'] * 0.35 +
    cadet_study['avg_progress'] * 0.45 +
    (cadet_study['practice_tests'] / cadet_study['practice_tests'].max().replace(0, 1)) * 20
).clip(0, 100)

weak_subjects = toga_study[(toga_study['study_progress_pct'] < 70) | (toga_study['avg_quiz_score'] < 70)]
weak_subjects = weak_subjects.groupby('cadet_id')['subject'].apply(lambda x: ', '.join(sorted(x.unique()))).reset_index()
cadet_study = cadet_study.merge(weak_subjects, on='cadet_id', how='left').fillna({'subject': 'None'})
cadet_study.rename(columns={'subject': 'weak_subjects'}, inplace=True)

cadet_study[['name', 'study_readiness', 'weak_subjects', 'inactivity_risk']].head()

## Finance Risk Analysis
Analyze outstanding balances, payment completion, payment risk, and training continuity risk.

In [ ]:
payments = payments.copy()
payments['payment_completion_pct'] = (payments['paid_amount'] / payments['invoiced_amount']).replace([np.inf, -np.inf], 0) * 100
payments['payment_risk_pct'] = (payments['outstanding_amount'] / payments['invoiced_amount']).replace([np.inf, -np.inf], 0) * 100

payments = payments.merge(cadets[['cadet_id', 'name', 'progress_pct']], on='cadet_id')
payments['continuity_risk'] = payments.apply(lambda row: 'High' if row['payment_risk_pct'] > 40 and row['progress_pct'] < 60 else ('Medium' if row['payment_risk_pct'] > 20 else 'Low'), axis=1)

payment_summary = payments.sort_values('payment_risk_pct', ascending=False).head(10)
payment_summary[['name', 'payment_completion_pct', 'payment_risk_pct', 'continuity_risk']].head()

## Explainable Cadet Risk Scoring
Combine flight progress, study progress, quiz performance, inactivity, payment risk, cancellations, and delays into a transparent risk score.

In [ ]:
cancellations = sorties[sorties['status'] == 'cancelled'].groupby('cadet_id').size().rename('cancel_count').reset_index()
avg_delay = actual_flights.groupby('cadet_id')['delay_minutes'].mean().rename('avg_delay').reset_index()

cadet_risk = cadets[['cadet_id', 'name', 'progress_pct']].merge(
    cadet_study[['cadet_id', 'avg_progress', 'avg_quiz', 'study_readiness', 'inactive_days', 'weak_subjects']],
    on='cadet_id'
).merge(
    payments[['cadet_id', 'payment_risk_pct', 'outstanding_amount']],
    on='cadet_id'
).merge(cancellations, on='cadet_id', how='left').merge(avg_delay, on='cadet_id', how='left')

cadet_risk['cancel_count'] = cadet_risk['cancel_count'].fillna(0)
cadet_risk['avg_delay'] = cadet_risk['avg_delay'].fillna(0)

cadet_risk['flight_risk'] = np.where(cadet_risk['progress_pct'] < 60, (60 - cadet_risk['progress_pct']) * 0.8, 0)
cadet_risk['study_risk'] = np.where(cadet_risk['avg_progress'] < 70, (70 - cadet_risk['avg_progress']) * 0.6, 0)
cadet_risk['quiz_risk'] = np.where(cadet_risk['avg_quiz'] < 75, (75 - cadet_risk['avg_quiz']) * 0.55, 0)
cadet_risk['inactivity_risk_score'] = np.where(cadet_risk['inactive_days'] > 14, (cadet_risk['inactive_days'] - 14) * 0.45, 0)
cadet_risk['payment_risk_score'] = cadet_risk['payment_risk_pct'] * 0.4
cadet_risk['cancel_risk_score'] = cadet_risk['cancel_count'] * 2.5
cadet_risk['delay_risk_score'] = np.where(cadet_risk['avg_delay'] > 20, (cadet_risk['avg_delay'] - 20) * 0.3, 0)

cadet_risk['risk_score'] = (
    cadet_risk['flight_risk'] * 0.27 +
    cadet_risk['study_risk'] * 0.18 +
    cadet_risk['quiz_risk'] * 0.20 +
    cadet_risk['inactivity_risk_score'] * 0.12 +
    cadet_risk['payment_risk_score'] * 0.13 +
    cadet_risk['cancel_risk_score'] * 0.06 +
    cadet_risk['delay_risk_score'] * 0.04
).clip(0, 100).round(1)

cadet_risk['risk_level'] = pd.cut(cadet_risk['risk_score'], bins=[-1, 39, 69, 100], labels=['Low', 'Medium', 'High'])

risk_drivers = []
for _, row in cadet_risk.iterrows():
    drivers = []
    if row['progress_pct'] < 60:
        drivers.append('Low flight progress')
    if row['avg_progress'] < 70:
        drivers.append('Weak study progress')
    if row['avg_quiz'] < 75:
        drivers.append('Below-target quiz score')
    if row['inactive_days'] > 30:
        drivers.append('Inactivity risk')
    if row['payment_risk_pct'] > 25:
        drivers.append('High outstanding payment')
    if row['cancel_count'] >= 2:
        drivers.append('Repeated cancellations')
    if row['avg_delay'] > 30:
        drivers.append('High average delay')
    risk_drivers.append(', '.join(drivers) if drivers else 'No dominant risk driver')

cadet_risk['top_risk_drivers'] = risk_drivers
risk_export = cadet_risk[['cadet_id', 'name', 'risk_score', 'risk_level', 'top_risk_drivers']]
risk_export.to_csv('data/risk_scores.csv', index=False)

cadet_risk.head()

## Visualizations
Generate business-facing charts and save them to the `charts/` folder.

In [ ]:
import os
os.makedirs('charts', exist_ok=True)

def save_fig(fig, path):
    """Save figure as PNG if kaleido is available, else HTML."""
    try:
        fig.write_image(path)
        print(f'Saved PNG: {path}')
    except Exception as e:
        html_path = path.replace('.png', '.html')
        fig.write_html(html_path)
        print(f'kaleido unavailable ({e}); saved HTML: {html_path}')

fig = px.bar(aircraft_util, x='registration', y='utilization_pct',
             title='Aircraft Utilization (%)',
             labels={'registration': 'Aircraft', 'utilization_pct': 'Utilization (%)'})
fig.update_layout(xaxis_tickangle=-45)
save_fig(fig, 'charts/aircraft_utilization.png')

fig = px.pie(top_cancellation_reasons, values='count', names='reason',
             title='Cancellation Reasons')
save_fig(fig, 'charts/cancellation_reasons.png')

fig = px.bar(cadets.sort_values('progress_pct'), x='name', y='progress_pct',
             title='Cadet Progress (%)',
             labels={'name': 'Cadet', 'progress_pct': 'Progress (%)'})
fig.update_layout(xaxis_tickangle=-45)
save_fig(fig, 'charts/cadet_progress.png')

fig = px.bar(cadet_study.sort_values('study_readiness'), x='name', y='study_readiness',
             title='Study Readiness Score',
             labels={'name': 'Cadet', 'study_readiness': 'Study Readiness'})
fig.update_layout(xaxis_tickangle=-45)
save_fig(fig, 'charts/study_readiness.png')

fig = px.bar(payments.sort_values('payment_risk_pct', ascending=False), x='name', y='payment_risk_pct',
             title='Payment Risk (%)',
             labels={'name': 'Cadet', 'payment_risk_pct': 'Payment Risk (%)'})
fig.update_layout(xaxis_tickangle=-45)
save_fig(fig, 'charts/payment_risk.png')

fig = px.histogram(cadet_risk, x='risk_score', nbins=20,
                   title='Cadet Risk Scores Distribution',
                   labels={'risk_score': 'Risk Score'})
save_fig(fig, 'charts/cadet_risk_scores.png')

flight_vs_study = cadet_study.merge(cadets[['cadet_id', 'progress_pct']], on='cadet_id')
fig = px.scatter(flight_vs_study, x='progress_pct', y='avg_progress', text='name',
                 title='Flight vs Study Progress',
                 labels={'progress_pct': 'Flight Progress (%)', 'avg_progress': 'Study Progress (%)'})
fig.update_traces(textposition='top center')
save_fig(fig, 'charts/flight_vs_study_progress.png')

print('Charts generation complete.')


## Executive Insights
Summarize the operational, training, TOGA, and finance findings into actionable recommendations for AIRMAN leadership.

In [ ]:
# Executive summary bullet points
top_insights = [
    f'Dispatch reliability is {dispatch_reliability:.1f}%; most operations are delivered, but delays still affect efficiency.',
    f'Cancellation rate is {cancellation_rate:.1f}% and is driven by Weather and Maintenance events.',
    f'{len(cadets_at_risk)} cadets are below 60% completion and present the highest training risk.',
    'TOGA readiness is weakest in low-progress subjects and among cadets with more than 30 inactive days.',
    'Outstanding payment balances correlate with continuity risk for under-performing cadets.',
]

for insight in top_insights:
    print('- ' + insight)

## Next Steps
1. Review `reports/` for narrative findings.
2. Use `charts/` visuals for presentation.
3. Refine risk and schedule monitoring based on additional weather and flight instructor feedback.